In [1]:
%%capture
# will take approx 5 min
import os
if not os.path.exists('open-images-bus-trucks'):
    %pip install torch_snippets ultralytics
    !wget --quiet https://www.dropbox.com/s/agmzwk95v96ihic/open-images-bus-trucks.tar.xz
    !tar -xf open-images-bus-trucks.tar.xz
    !rm open-images-bus-trucks.tar.xzd

In [2]:
from torch_snippets import *
def prepare_yolo_data(image_folder, labels_folder, train_txt, val_txt, output_folder):
    def prepare_yolo_split(split, image_folder, labels_folder, txt, output_folder):
        items = readlines(txt)
        for item in items:
            item = stem(item)
            im_to = output_folder/'images'/split/f'{item}.jpg'
            ann_to = output_folder/'labels'/split/f'{item}.txt'
            makedir(parent(im_to)), makedir(parent(ann_to))
            im = image_folder/f'{item}.jpg'
            ann = labels_folder/f'{item}.txt'
            im.cp(im_to)
            ann.cp(ann_to)
    prepare_yolo_split('train',image_folder,labels_folder,train_txt,output_folder)
    prepare_yolo_split('val',image_folder,labels_folder,val_txt,output_folder)

data_version = 'all'
yolo_path = P(f'./yolo-open-images-bus-trucks-{data_version}')
try: yolo_path.rmtree(force=True)
except: ...
prepare_yolo_data(
    P('open-images-bus-trucks/images'),
    P(f'open-images-bus-trucks/yolo_labels/{data_version}/labels/'),
    P(f'open-images-bus-trucks/yolo_labels/{data_version}/train.txt'),
    P(f'open-images-bus-trucks/yolo_labels/{data_version}/val.txt'),
    P(yolo_path)
)

[06/08/26 01:12:55] INFO     loaded 12439 lines                                                                                           ]8;id=769712;file:///tmp/ipykernel_924/2786636697.py:4\2786636697.py]8;;\:]8;id=370881;file:///tmp/ipykernel_924/2786636697.py:4#prepare_yolo_split:4\prepare_yolo_split:4]8;;\

[06/08/26 01:12:59] INFO     loaded 5104 lines                                                                                            ]8;id=521938;file:///tmp/ipykernel_924/2786636697.py:4\2786636697.py]8;;\:]8;id=839224;file:///tmp/ipykernel_924/2786636697.py:4#prepare_yolo_split:4\prepare_yolo_split:4]8;;\

In [3]:
%%writefile bus_truck_all.yaml

path: /content/yolo-open-images-bus-trucks-all/ # dataset root dir
train: images/train # train images (relative to 'path') 4 images
val: images/val # val images (relative to 'path') 4 images

# Classes
names:
  0: bus
  1: truck

Writing bus_truck_all.yaml


In [4]:
from ultralytics import YOLO

# Load a model
model = YOLO("yolov8m.pt")  # load a pretrained model (recommended for training)
_ = model.train(data=f"bus_truck_{data_version}.yaml", epochs=3)  # train the model

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.61 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=bus_truck_all.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, in

In [5]:
# results = model("https://ultralytics.com/images/bus.jpg")  # predict on an image
val_images = Glob("/content/yolo-open-images-bus-trucks-mini/images/val/")

for _ in range(10):
    results = model(choose(val_images))
    boxes = results[0].boxes.xyxy.cpu().numpy().astype(int)
    classes = [results[0].names[cls] for cls in results[0].boxes.cls.cpu().numpy()]
    show(results[0].orig_img, bbs=boxes, texts=classes, sz=5, text_sz=10)

ValueError: high <= 0